# SmartBite: the plate as the ruler for grams
Finds the plate (OWLv2 + SAM 2.1), undoes the camera tilt and scales by the plate. Benchmarks the first portion model with and without plate scaling on weighed Nutrition5k test dishes (overhead, zoomed, side-angle stills), then trains a plate-relative portion model on plate-only views.

In [ ]:
# SmartBite food recognition - training run (Kaggle GPU or Colab)
import os, sys, glob, json, shutil, subprocess, zipfile, time
CODE_SLUG, DATA_CFG, CLASSES_CFG, TRAIN_EXTRA = "smartbite-cv-code-v3", "configs/data_v2.yaml", "configs/classes_v2.yaml", ""
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working/SmartBite" if ON_KAGGLE else "/content/SmartBite"
OUT = "/kaggle/working/outputs" if ON_KAGGLE else "/content/outputs"
CACHE = "/tmp/smartbite_cache"

def run(cmd):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=WORK)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

if ON_KAGGLE and not os.path.exists(f"{WORK}/smartbite_cv"):
    # The private code dataset is either auto-extracted or still a zip; handle both. Look in its own
    # mount first: a recursive search over every attached image dataset takes minutes.
    # Mounts are /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>.
    mounts = glob.glob(f"/kaggle/input/{CODE_SLUG}") + glob.glob(f"/kaggle/input/*/*/{CODE_SLUG}")
    inits = [i for m in mounts for i in glob.glob(f"{m}/**/smartbite_cv/__init__.py", recursive=True)]
    if inits:
        shutil.copytree(os.path.dirname(os.path.dirname(inits[0])), WORK, dirs_exist_ok=True)
    else:
        zips = [z for m in mounts for z in glob.glob(f"{m}/**/smartbite_code.zip", recursive=True)]
        assert zips, "attach the smartbite-cv-code dataset"
        zipfile.ZipFile(zips[0]).extractall(WORK)
# Colab: upload/clone the repo to /content/SmartBite and put the raw datasets under
# /content/SmartBite/data/raw/<same folder names as configs/data.yaml>, then set PROFILE = "local".
PROFILE = "kaggle" if ON_KAGGLE else "local"
os.makedirs(OUT, exist_ok=True)
# No internet needed: timm comes from the bundled wheel, ImageNet weights from offline/*.safetensors.
whl = glob.glob(f"{WORK}/offline/timm-*.whl")
if ON_KAGGLE and not whl:
    raise RuntimeError("offline/ missing: the attached smartbite-cv-code dataset version is stale; re-run")
run(f"pip install -q --no-index --no-deps {whl[0]}" if whl else "pip install -q timm onnx onnxruntime")
run("nvidia-smi || true")
run("python -c \"import torch, timm; print('torch', torch.__version__, 'timm', timm.__version__)\"")
import torch
assert torch.cuda.is_available(), "No GPU attached: enable the GPU accelerator (Kaggle needs a phone-verified account)"
try:
    import onnxruntime  # noqa: F401
    HAVE_ORT = True
except ImportError:
    HAVE_ORT = False  # ONNX export + latency are then done locally from best.pt
print("onnxruntime available:", HAVE_ORT)


In [ ]:
# Plate as the ruler (needs internet: Hugging Face weights, Google's Nutrition5k bucket).
import csv, random
run('pip install -q -U "transformers>=4.56" onnx onnxruntime')
run("python -c \"import transformers, onnxruntime; from transformers import Sam2Model, Owlv2ForObjectDetection; print('transformers', transformers.__version__)\"")
HAVE_ORT = True
marker = [m for depth in range(4) for m in glob.glob("/kaggle/input/" + "*/" * depth + "imagery/realsense_overhead")]
assert marker, "attach a Nutrition5k mirror with imagery/realsense_overhead"
N5K = os.path.dirname(os.path.dirname(marker[0]))
SPLITS = f"{WORK}/data/raw/nutrition5k/dish_ids/splits"
P = f"{OUT}/plate"
os.makedirs(P, exist_ok=True)
t0 = time.time()
# 0. Overhead + zoom detections from the previous run (same detector, same images; side stills are redone).
seed = f"{WORK}/data/plate_cache/detections_overhead.jsonl"
if os.path.exists(seed):
    shutil.copy(seed, f"{P}/detections.jsonl")
# 1. One complete still per side-angle camera: HTTP range request for the head of each video (official bucket).
for split, d in (("depth_test", "/tmp/side_test"), ("depth_train", "/tmp/side_train")):
    run(f"python tools/fetch_side_frames.py --ids {SPLITS}/{split}_ids.txt --out {d} --workers 32")
    shutil.copy(f"{d}/frames.csv", f"{P}/frames_{split}.csv")
print(f"stills: {(time.time() - t0) / 60:.1f} min", flush=True)
# 1b. Plate detection for every uncached photo, one process per GPU, then one merged cache.
gpus = max(1, torch.cuda.device_count())
src = f"--n5k {N5K} --side-train /tmp/side_train --side-test /tmp/side_test"
for i in range(gpus):   # every shard starts from the same cache, so the shards split the same to-do list
    if os.path.exists(f"{P}/detections.jsonl"):
        shutil.copy(f"{P}/detections.jsonl", f"/tmp/det_{i}.jsonl")
run("(" + " ".join(f"CUDA_VISIBLE_DEVICES={i} python -m smartbite_cv.plate_portion prefill {src} --cache /tmp/det_{i}.jsonl "
                   f"--shard {i}/{gpus} & p{i}=$!;" for i in range(gpus))
    + " " + " && ".join(f"wait $p{i}" for i in range(gpus)) + ")")
recs = {}
for i in range(gpus):
    for line in (open(f"/tmp/det_{i}.jsonl") if os.path.exists(f"/tmp/det_{i}.jsonl") else []):
        if line.strip():
            r = json.loads(line)
            recs[r["key"]] = r
with open(f"{P}/detections.jsonl", "w") as f:
    f.writelines(json.dumps(r) + chr(10) for r in recs.values())
print(f"detections: {len(recs)} cached ({(time.time() - t0) / 60:.1f} min)", flush=True)
# 2. First portion model on test photos, as is and with plate scaling (reference plate from training photos).
run(f"python -m smartbite_cv.plate_bench --n5k {N5K} --side /tmp/side_test --portion models/food_weight_finder_no_plate.onnx "
    f"--cache {P}/detections.jsonl --out {P}/bench")
print(f"benchmark: {(time.time() - t0) / 60:.1f} min", flush=True)
# 3. Plate-only views (detections cached), plate-relative model, test split used once, ONNX export.
run(f"python -m smartbite_cv.plate_portion views --n5k {N5K} --side-train /tmp/side_train --side-test /tmp/side_test "
    f"--cache {P}/detections.jsonl --out /tmp/views")
shutil.copy("/tmp/views/views.csv", P)
print(f"views: {(time.time() - t0) / 60:.1f} min", flush=True)
weights = ("--set model.pretrained_file=offline/efficientnet_b0.ra_in1k.safetensors"
           if os.path.exists(f"{WORK}/offline/efficientnet_b0.ra_in1k.safetensors") else "")
run(f"python -m smartbite_cv.plate_portion train --views /tmp/views --config configs/train/plate_portion.yaml {weights} --out runs/plate_portion")
run(f"python -m smartbite_cv.plate_portion evaluate --run runs/plate_portion --views /tmp/views --old-preds {P}/bench/predictions.csv")
run("python -m smartbite_cv.plate_portion export --run runs/plate_portion --out artifacts/plate_portion")
shutil.copytree(f"{WORK}/artifacts/plate_portion", f"{P}/artifact", dirs_exist_ok=True)
for f in ["best.pt", "history.csv", "config_resolved.json"]:
    shutil.copy(f"{WORK}/runs/plate_portion/{f}", P)
shutil.copytree(f"{WORK}/runs/plate_portion/eval", f"{P}/eval", dirs_exist_ok=True)
rows = [r for r in csv.DictReader(open("/tmp/views/views.csv")) if r["file"] and r["split"] == "test"]
os.makedirs(f"{P}/views_sample", exist_ok=True)
for r in random.Random(1).sample(rows, min(40, len(rows))):
    shutil.copy(f"/tmp/views/{r['file']}", f"{P}/views_sample/")
print(f"total: {(time.time() - t0) / 60:.1f} min")
if ON_KAGGLE:
    shutil.rmtree(WORK, ignore_errors=True)
print(open(f"{P}/bench/summary.md").read())
print(open(f"{P}/eval/summary.md").read())
